# Updating MEI Metadata and Correcting Ficta

## Import Libraries

In [1]:
from bs4 import BeautifulSoup
import uuid
import glob
import pandas as pd
import os
from datetime import datetime
import chardet
import random

## Create Local Folder for MEI Files

In [2]:

    
MUSDIR = ("Music_Files")
CHECK_FOLDER = os.path.isdir(MUSDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MUSDIR)
    print("created folder : ", MUSDIR)
else:
    print(MUSDIR, "folder already exists.")

Music_Files folder already exists.


## Various Ways to Import MEI Files

In [14]:
# Various paths to import files

# folder = "/Users/rfreedma/Documents/CRIM_Python/crim-local/CRIM-online/crim/static/mei/MEI_4.0/*.mei"

# piece = "/Users/rfreedma/Desktop/OK/CRIM_Mass_0031_1.mei"

file_list = glob.glob('Music_Files/*')
# file_name = os.path.basename(file_list[0])

file_list


['Music_Files/Daser_Missa_Ecce_nunc_Kyrie_E.mei']

### Chose Which Modules are Active for MEI File Updates

In [15]:

correct_ficta = True
update_metadata = False
add_voice_labels = True

## Specify Location of Metadata to Use for the Updates of MEI

More about the metadata. This code updates the following fields in the MEI:

- the respStmt, including names of composers and editors, with 'role' attributes, and with 'auth' and 'auth.uri' attributes for the composer
- the titleStmt, including the title of the musical work
- the workList, including the composer, work title, and genre (which is preserved under <classification>)
- manifestationList, including the standard identifier (such as RISM) for the source, the title and date of the source, and up to two 'publishers' along with their 'auth' and 'auth.uri' attributes.  Also the city and institution where the source is to be found, and finally any shelf number of the source
-the pubStmt, where we record the overall location of this encoded file (such as The CRIM Project) along with the copyright owners, the rights statement, and the date on which this digital file was produced

In [16]:
model_metadata = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vQcKD0Lb3zw7E8El1vRf6cduN03_ZvwLlMkDf6azvQrFHVt3xdyOzgJOtuu4WzqYWPpQHhaUMkkpF7V/pub?output=csv'
df = pd.read_csv(model_metadata).fillna('')

# Convert the DataFrame to a dictionary
metadata_dict = df.to_dict(orient='records')

## Run the Updates

In [17]:
# Get each file in the folder, and collect the file and the name
for filename in file_list:
    with open(filename, 'rb') as file:
        # Detect the encoding
        result = chardet.detect(file.read())
        # Seek to the beginning of the file
        file.seek(0)
        # Decode the file using the detected encoding
        content = file.read().decode(result['encoding'])
        file_name = os.path.basename(filename)
        # print(file_name)
        base_name, extension = os.path.splitext(file_name)        
        revised_name = base_name + "_rev" + extension
        soup = BeautifulSoup(content, 'xml')

        # correct the ficta
        if correct_ficta == True:
            # remove the 'dir' tags created by sib MEI:
            dir_tags = soup.find_all('dir')
            # Remove each 'dir' tag from the soup
            for tag in dir_tags:
                tag.decompose()
            # get the colored notes:
            notes = soup.find_all('note', attrs={'color': True})
            # Find the note tag and remove the color attribute
            for note in notes:
                del note['color']
                accid_value = note.find('accid')['accid.ges']
                accid_xml = note.find('accid')['xml:id']
                random_id = random.randint(10000,  99999)
                # # create new supplied parent tag for accid
                supplied_tag = soup.new_tag('supplied', attrs={'reason' : 'edit', 
                                                               'xml:id': random_id})                                                 
                # create the new accid tag and make it a child of supplied
                new_accid_tag = soup.new_tag('accid', accid=accid_value, func="edit", place="above", xmlid="m-607")
                supplied_tag.append(new_accid_tag)
                # replace the old accid tag with the supplied + accid
                accid_tag = note.find('accid')
                accid_tag.replace_with(supplied_tag)
                
        # add voice names to staffDef as 'label' attribute
        if add_voice_labels == True:
            staff_defs = soup.find_all('staffDef')
            for staff_def in staff_defs:
                # Find the nested <label> tag
                label = staff_def.find('label')
                # Extract the text content of the <label> tag
                label_text = label.get_text() if label else None
                # Add the extracted text as an attribute 'label' to the <staffDef> tag
                if label_text:
                    staff_def['label'] = label_text

                    # Iterate over each <staffDef> tag
            for staff_def in staff_defs:
                # Find the nested <label> tag
                label = staff_def.find('label')
                # Extract the text content of the <label> tag
                label_text = label.get_text() if label else None
                # Extract the first letter, capitalize it
                if label_text:
                    first_letter = label_text[0].upper()
                    # Find the nested <labelAbbr> tag
                    label_abbr = staff_def.find('labelAbbr')
                    # Update the text content of the <labelAbbr> tag
                    if label_abbr:
                        label_abbr.string = first_letter + '.'
        
        # update metadata
        if update_metadata == True:
            # get the matching dictionar for this piece
            matching_dict = next((item for item in metadata_dict if item['CRIM_Model_Mei'] == file_name), None)

            # get the composer from the dict; supply default value in case it's missing in the dict
            composer_csv = matching_dict.get('ComposerName', '')
            # get respStmt 
            respStmt = soup.find('respStmt')
            # remove all of the persName tags from the respStmt
            [person.decompose() for person in respStmt.find_all('persName')]

            # add composer
            comp_tag = soup.new_tag('persName')
            comp_tag.string = composer_csv
            comp_tag['role'] = 'composer'
            comp_tag['auth'] = 'VIAF'
            comp_tag['auth.uri'] = matching_dict.get('Composer VIAF', '')
            respStmt.append(comp_tag)

            # also add composer in the workList, too
            comp = soup.find('workList').find('work').find('composer')
            comp.string = composer_csv
            comp['auth'] = 'VIAF'
            comp['auth.uri'] = matching_dict.get('Composer VIAF', '')

            # add editors
            editors_csv = matching_dict.get('Editor', '')
            editor_names_list = editors_csv.split("|") # Splits the string at "|"
            editor_names = [name.strip() for name in editor_names_list]
            for editor in editor_names:
                ed_tag = soup.new_tag('persName')
                ed_tag.string = editor
                ed_tag['role'] = 'editor'
                respStmt.append(ed_tag)

            # work title
            work_title_csv = matching_dict.get('Model Title', '')
            soup.find('titleStmt').find('title').string = work_title_csv

            # and update title in workList, too
            soup.find('workList').find('work').find('title').string = work_title_csv

            # date is handled as 'event' tag in MEI, so we will skip this for now, 
            # since it is not very useful for 16th C
            # instead we record the source date below
            # but here is how to get the data from the CSV
            piece_date = matching_dict.get('Piece Date', '')
            

            # work genre
            work_genre_csv = matching_dict.get('Genre of Model', '')
            soup.find('work').find('classification').find('termList').find('term').string = work_genre_csv

            # source title
            source_csv = matching_dict.get('Source Title', '')
            soup.find('manifestationList').find('manifestation').find('titleStmt').find('title').string = source_csv

            # source publishers:  first remove ALL names tags from the pubStmt
            [person.decompose() for person in soup.find('manifestationList').find('pubStmt').find('publisher').find_all('persName')]

            # now deal with pub 1 and metadata
            source_pub_1_csv = matching_dict.get('Source Publisher 1')
            pub_1_tag = soup.new_tag('persName')
            pub_1_tag.string = source_pub_1_csv
            pub_1_tag['auth'] = 'VIAF'
            pub_1_tag['auth.uri'] = matching_dict.get('Publisher 1 VIAF', '')
            soup.find('manifestationList').find('pubStmt').find('publisher').append(pub_1_tag)

            # now deal with pub 2 and metadata
            source_pub_2_csv = matching_dict.get('Source Publisher 2')

            # add the new publisher 2
            pub_2_tag = soup.new_tag('persName')
            pub_2_tag.string = source_pub_2_csv
            pub_2_tag['auth'] = 'VIAF'
            pub_2_tag['auth.uri'] = matching_dict.get('Publisher 2 VIAF', '')
            soup.find('manifestationList').find('pubStmt').find('publisher').append(pub_2_tag)

            # source date
            source_date_csv = matching_dict.get('Source Date')
            date_tag = soup.find('manifestationList').find('pubStmt').find('date')
            date_tag.string = source_date_csv
            if 'isodate' in date_tag.attrs:
                del date_tag['isodate']

            # source reference
            # here we need to specify the kind of reference number, based on RISM B or Census Catalog
            source_reference_csv = matching_dict.get('Source Reference')
            if 'RISM' in source_reference_csv:
                soup.find('manifestationList').find('manifestation').find('identifier').string = source_reference_csv
                soup.find('manifestationList').find('manifestation').find('identifier')['type'] = 'RISM'
            if 'CENSUS' in source_reference_csv:

                soup.find('manifestationList').find('manifestation').find('identifier').string = source_reference_csv
                soup.find('manifestationList').find('manifestation').find('identifier')['type'] = 'CENSUS'
            else: 
                soup.find('manifestationList').find('manifestation').find('identifier').string = source_reference_csv
                soup.find('manifestationList').find('manifestation').find('identifier')['type'] = ''

            # source location
            source_location_csv = matching_dict.get('Source Location')
            repository_tag = soup.find('manifestationList').find('manifestation').find('physLoc').find('repository')
            repository_tag.append(soup.new_tag('geogName'))
            repository_tag.find('geogName').string = source_location_csv

            # source institution
            source_institution_csv = matching_dict.get('Source Institution')
            soup.find('manifestationList').find('manifestation').find('physLoc').find('repository').find('corpName').string = source_institution_csv

            # source shelfmark
            source_shelfmark_csv = matching_dict.get('Source Shelfmark')
            soup.find('manifestationList').find('manifestation').find('physLoc').find('identifier').string = source_shelfmark_csv

            # publisher of this encoding.  For CRIM, this is hard coded here, not in the CSV
            soup.find('pubStmt').find('publisher').string = "Citations: The Renaissance Imitation Mass"
            soup.find('pubStmt').find('publisher')['uri'] = 'crimproject.org'

            # rights
            rights_csv = matching_dict.get('Rights Statement')
            soup.find('pubStmt').find('availability').string = rights_csv

            # owner
            owners_csv = matching_dict.get('Copyright Owner(s)')
            # remove all existing owner distributors
            [dist.decompose for dist in soup.find('pubStmt').find_all('distributor')]
            # get new owner information
            owner_list = owners_csv.split("|") # Splits the string at "|"
            owner_names = [name.strip() for name in owner_list]
            for owner in owner_names:
                owner_tag = soup.new_tag('persName')
                owner_tag.string = owner
                soup.find('pubStmt').append(ed_tag)

            # date of encoding update
            current_date = current_date = datetime.now().date().isoformat()

            # Add the 'currentDate' attribute to the 'corpName' tag
            soup.find('pubStmt').find('date')['isodate'] = current_date

    #write file (must make sure folder exists!)   
    folder_path = "Updates/"
    file_path = os.path.join(folder_path, file_name)
    pretty_xml = soup.prettify()
    with open(file_path, 'w') as f:
        f.write(str(pretty_xml))

